<a href="https://colab.research.google.com/github/RosdianaPutri01/praktikum_sistempakar/blob/main/SISTEMPAKAR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install experta

  Preparing metadata (setup.py) ... done
  Created wheel for frozendict: filename=frozendict-1.2-py3-none-any.whl size=3149 sha256=38cf10b5ec000cadc36ab936a985bba032e635d4ed2b4d93e77a261af93d768f
  Stored in directory: /root/.cache/pip/wheels/49/ac/f8/cb8120244e710bdb479c86198b03c7b08c3c2d3d2bf448fd6e
Successfully built frozendict
  Attempting uninstall: frozendict
    Found existing installation: frozendict 2.4.6
    Uninstalling frozendict-2.4.6:
      Successfully uninstalled frozendict-2.4.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
yfinance 0.2.55 requires frozendict>=2.3.4, but you have frozendict 1.2 which is incompatible.


In [ ]:
!pip install --upgrade frozendict

  Attempting uninstall: frozendict
    Found existing installation: frozendict 1.2
    Uninstalling frozendict-1.2:
      Successfully uninstalled frozendict-1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
experta 1.9.4 requires frozendict==1.2, but you have frozendict 2.4.6 which is incompatible.


In [8]:
from experta import *

class Diagnosis(KnowledgeEngine):

  @Rule(Fact(cough=True) & Fact(fever=True) & Fact(fatigue=True))
  def flu(self):
    print("Diagnosis: You may have the Flu.")

  @Rule(Fact(cough=True) & Fact(fever=True) & Fact(breathing_difficulty=True))
  def pneumonia(self):
    print("Diagnosis: You may have Pneumonia.")

  @Rule(Fact(sneezing=True) & Fact(runny_nose=True) & Fact(cough=False))
  def cold(self):
    print("Diagnosis: You may have a Common Cold.")

  @Rule(Fact(sore_throat=True) & Fact(fever=True))
  def throat_infection(self):
    print("Diagnosis: You may have a Throat Infection.")

  @Rule(NOT(Fact(cough=True)) & NOT(Fact(fever=True)) & NOT(Fact(fatigue=True)) & NOT(Fact(breathing_difficulty=True)) & NOT(Fact(sneezing=True)) & NOT(Fact(runny_nose=True)) & NOT(Fact(sore_throat=True)))
  def healthy(self):
    print("Diagnosis: You seem healthy.")

def get_input():
  """Helper function to get user input and convert to boolean (yes/no)."""
  def ask_question(question):
    while True:
      answer = input(question + " (yes/no): ").strip().lower()
      if answer in ["yes", "no"]:
        return answer == "yes"
      else:
        print("Invalid input. Please answer with 'yes' or 'no'.")

  return {
      "cough": ask_question("Do you have a cough?"),
      "fever": ask_question("Do you have a fever?"),
      "fatigue": ask_question("Do you feel tired or fatigued?"),
      "breathing_difficulty": ask_question("Do you have difficulty breathing?"),
      "sneezing": ask_question("Are you sneezing?"),
      "runny_nose": ask_question("Do you have a runny nose?"),
      "sore_throat": ask_question("Do you have a sore throat?")
  }

# running the expert system
if __name__ == "__main__":
  symptoms = get_input()
  engine = Diagnosis()
  engine.reset() # reset the knowledge engine

  for symptom, present in symptoms.items():
    engine.declare(Fact(**{symptom: present}))

  engine.run() # run the inference engineyes

Do you have a cough? (yes/no): yes
Do you have a fever? (yes/no): yes
Do you feel tired or fatigued? (yes/no): yes
Do you have difficulty breathing? (yes/no): no
Are you sneezing? (yes/no): no
Do you have a runny nose? (yes/no): yes
Do you have a sore throat? (yes/no): yes
Diagnosis: You may have a Throat Infection.
Diagnosis: You may have the Flu.


In [9]:
from experta import *

class SistemPakarMedis(KnowledgeEngine):
  @Rule(Fact(demam=True) & Fact(batuk=True))
  def flu(self):
    print("Diagnosis: Flu.")

  @Rule(Fact(Sakit_Tenggorokan=True) & Fact(demam=True))
  def throat_infection(self):
    print("Diagnosis: Radang Tenggorokan.")

  @Rule(Fact(nyeri_otot=True) & Fact(nyeri_perut=True))
  def hernia(self):
    print("Diagnosis: Hernia. Innalillahi")



engine = SistemPakarMedis()
engine.reset()
engine.declare(Fact(nyeri_otot=True))
engine.declare(Fact(nyeri_perut=True))  # Input symptoms
engine.run()


Diagnosis: Hernia. Innalillahi


In [10]:
def forward_chaining(facts,rules):
    inferred = set (facts)
    changed = True

    while changed:
        changed = False
        for rule in rules:
            if rule["if"].issubset(inferred) and rule["then"] not in inferred:
                inferred.add(rule["then"])
                changed = True
    return inferred

facts = {"has_feathers","has_beak","carnivore"}
rules = [
    {"if":{"has_feathers","has_beak"},"then":"is_bird"},
    {"if":{"lays_eggs","is_bird"},"then":"is_chicken"},
    {"if":{"cannot_fly","is_bird"},"then":"is_penguin"},
    {"if":{"carnivore","is_bird"},"then":"is_eagle"}

]

result = forward_chaining(facts,rules)
print("Inferred facts:",result)


Inferred facts: {'has_feathers', 'is_eagle', 'carnivore', 'has_beak', 'is_bird'}


In [11]:
def backward_chaining(goal, facts, rules):
    if goal in facts:
        return True
    for rule in rules:
        if rule["then"] == goal:
            if all(backward_chaining(cond, facts, rules) for cond in rule["if"]):
                return True
    return False

facts = {"likes_computers", "solves_problems", "likes_to_design"}
rules = [
    {"if": {"likes_computers", "solves_problems"}, "then": "should_be_engineer"},
    {"if": {"should_be_engineer", "likes_programming"}, "then": "software_engineer"},
    {"if": {"should_be_engineer", "likes_to_design"}, "then": "UI/UX_engineer"},
]

goal = "UI/UX_engineer"
result = backward_chaining(goal, facts, rules)
print(f"Is '{goal}' provable? -> {result}")

Is 'UI/UX_engineer' provable? -> True


In [12]:
def forward_chaining(facts, rules):
    inferred = set(facts)
    changed = True
    while changed:
        changed = False
        for rule in rules:
            if rule["if"].issubset(inferred) and rule["then"] not in inferred:
                inferred.add(rule["then"])
                changed = True
    return inferred

facts = {"has_wheels", "has_engine", "has_four_wheels"}
rules = [
    {"if": {"has_wheels", "has_engine"}, "then": "is_vehicle"},
    {"if": {"is_vehicle", "has_two_wheels"}, "then": "is_motorcycle"},
    {"if": {"is_vehicle", "has_four_wheels"}, "then": "is_car"}
]


result = forward_chaining( facts, rules)
print("Inferred facts:", result)

Inferred facts: {'has_engine', 'is_car', 'is_vehicle', 'has_four_wheels', 'has_wheels'}


In [13]:
def backward_chaining(goal, facts, rules):
    if goal in facts:
        return True
    for rule in rules:
        if rule["then"] == goal:
            if all(backward_chaining(cond, facts, rules) for cond in rule["if"]):
                return True
    return False

facts = {"has_feathers", "has_small_wings"}
rules = [
    {"if": {"is_bird", "cannot_fly"}, "then": "is_penguin"},
    {"if": {"has_feathers"}, "then": "is_bird"},
    {"if": {"has_small_wings"}, "then": "cannot_fly"},
]

goal = "is_penguin"
result = backward_chaining(goal, facts, rules)
print(f"Is '{goal}' provable? -> {result}")

Is 'is_penguin' provable? -> True


In [ ]:
from experta import *

class Diagnosis(KnowledgeEngine):

    @Rule(Fact(batuk=True) & Fact(demam=True) & Fact(lelah=True))
    def bronkitis(self):
        print("Diagnosa: Anda mungkin menderita Bronkitis.")

    @Rule(Fact(batuk=True) & Fact(demam=True) & Fact(sesak_napas=True))
    def pneumonia(self):
        print("Diagnosa: Anda mungkin menderita Pneumonia.")

    @Rule(Fact(bersin=True) & Fact(hidung_meler=True) & Fact(batuk=False))
    def rhinitis(self):
        print("Diagnosa: Anda mungkin menderita Rhinitis Alergi.")

    @Rule(Fact(tenggorokan_sakit=True) & Fact(demam=True))
    def faringitis(self):
        print("Diagnosa: Anda mungkin menderita Faringitis.")

    @Rule(Fact(demam=True) & Fact(batuk=True) & Fact(kehilangan_penciuman=True))
    def covid19(self):
        print("Diagnosa: Anda mungkin menderita COVID-19.")

    @Rule(Fact(batuk=True) & Fact(sesak_napas=True) & Fact(mengi=True))
    def asma(self):
        print("Diagnosa: Anda mungkin menderita Asma.")

    @Rule(Fact(bersin=True) & Fact(hidung_meler=True) & Fact(mata_gatal=True))
    def alergi(self):
        print("Diagnosa: Anda mungkin mengalami Reaksi Alergi.")

    @Rule(Fact(batuk=True) & Fact(demam=True) & Fact(berkeringat_malam=True) & Fact(batuk_berdarah=True))
    def tuberculosis(self):
        print("Diagnosa: Anda mungkin menderita Tuberkulosis (TBC).")

    @Rule(Fact(mual=True) & Fact(muntah=True) & Fact(diare=True) & Fact(demam=True))
    def demam_tifoid(self):
        print("Diagnosa: Anda mungkin menderita Demam Tifoid (Tipes).")

    @Rule(Fact(sakit_kepala=True) & Fact(nyeri_wajah=True) & Fact(hidung_meler=True))
    def sinusitis(self):
        print("Diagnosa: Anda mungkin menderita Sinusitis.")

    @Rule(Fact(sakit_kepala=True) & Fact(mual=True) & Fact(sensitif_cahaya=True))
    def migrain(self):
        print("Diagnosa: Anda mungkin menderita Migrain.")

    @Rule(Fact(demam=True) & Fact(ruam=True) & Fact(nyeri_otot=True))
    def demam_berdarah(self):
        print("Diagnosa: Anda mungkin menderita Demam Berdarah Dengue (DBD).")

    @Rule(Fact(tenggorokan_sakit=True) & Fact(sulit_menelan=True) & Fact(demam=True))
    def tonsilitis(self):
        print("Diagnosa: Anda mungkin menderita Tonsilitis (Radang Amandel).")

    @Rule(NOT(Fact(batuk=True)) & NOT(Fact(demam=True)) & NOT(Fact(lelah=True)) &
          NOT(Fact(sesak_napas=True)) & NOT(Fact(bersin=True)) &
          NOT(Fact(hidung_meler=True)) & NOT(Fact(tenggorokan_sakit=True)) &
          NOT(Fact(kehilangan_penciuman=True)) & NOT(Fact(mengi=True)) &
          NOT(Fact(mata_gatal=True)) & NOT(Fact(berkeringat_malam=True)) &
          NOT(Fact(batuk_berdarah=True)) & NOT(Fact(mual=True)) & NOT(Fact(muntah=True)) &
          NOT(Fact(diare=True)) & NOT(Fact(sakit_kepala=True)) & NOT(Fact(nyeri_wajah=True)) &
          NOT(Fact(sensitif_cahaya=True)) & NOT(Fact(ruam=True)) & NOT(Fact(nyeri_otot=True)) &
          NOT(Fact(sulit_menelan=True)))
    def sehat(self):
        print("Diagnosa: Anda tampaknya sehat.")

def ambil_input():
    def tanya(pertanyaan):
        while True:
            jawaban = input(pertanyaan + " (ya/tidak): ").strip().lower()
            if jawaban in ["ya", "tidak"]:
                return jawaban == "ya"
            else:
                print("Input tidak valid. Silakan jawab dengan 'ya' atau 'tidak'.")

    return {
        "batuk": tanya("Apakah Anda batuk?"),
        "demam": tanya("Apakah Anda demam?"),
        "lelah": tanya("Apakah Anda merasa lelah?"),
        "sesak_napas": tanya("Apakah Anda merasa sesak napas?"),
        "bersin": tanya("Apakah Anda bersin?"),
        "hidung_meler": tanya("Apakah hidung Anda meler?"),
        "tenggorokan_sakit": tanya("Apakah Anda sakit tenggorokan?"),
        "kehilangan_penciuman": tanya("Apakah Anda kehilangan indra penciuman?"),
        "mengi": tanya("Apakah Anda mengalami mengi (napas berbunyi tinggi)?"),
        "mata_gatal": tanya("Apakah mata Anda terasa gatal?"),
        "berkeringat_malam": tanya("Apakah Anda berkeringat di malam hari?"),
        "batuk_berdarah": tanya("Apakah Anda batuk berdarah?"),
        "mual": tanya("Apakah Anda merasa mual?"),
        "muntah": tanya("Apakah Anda muntah?"),
        "diare": tanya("Apakah Anda diare?"),
        "sakit_kepala": tanya("Apakah Anda sakit kepala?"),
        "nyeri_wajah": tanya("Apakah Anda merasakan nyeri di wajah atau sinus?"),
        "sensitif_cahaya": tanya("Apakah Anda sensitif terhadap cahaya?"),
        "ruam": tanya("Apakah Anda mengalami ruam kulit?"),
        "nyeri_otot": tanya("Apakah Anda merasakan nyeri otot?"),
        "sulit_menelan": tanya("Apakah Anda kesulitan menelan?")
    }

if __name__ == "__main__":
    print("Selamat datang di Sistem Pakar Diagnosa Penyakit")
    gejala = ambil_input()
    mesin = Diagnosis()
    mesin.reset()

    for gejala_item, ada in gejala.items():
        mesin.declare(Fact(**{gejala_item: ada}))

    mesin.run()

Selamat datang di Sistem Pakar Diagnosa Penyakit
Apakah Anda batuk? (ya/tidak): ya
Apakah Anda demam? (ya/tidak): ya
Apakah Anda merasa lelah? (ya/tidak): tidak
Apakah Anda merasa sesak napas? (ya/tidak): tidak
Apakah Anda bersin? (ya/tidak): ya
Apakah hidung Anda meler? (ya/tidak): tidak
Apakah Anda sakit tenggorokan? (ya/tidak): ya
Apakah Anda kehilangan indra penciuman? (ya/tidak): tidak
Apakah Anda mengalami mengi (napas berbunyi tinggi)? (ya/tidak): tidak
Apakah mata Anda terasa gatal? (ya/tidak): ya
Apakah Anda berkeringat di malam hari? (ya/tidak): tidak
Apakah Anda batuk berdarah? (ya/tidak): tidak
Apakah Anda merasa mual? (ya/tidak): tidak
Apakah Anda muntah? (ya/tidak): ya
Apakah Anda diare? (ya/tidak): tidak
Apakah Anda sakit kepala? (ya/tidak): ya
Apakah Anda merasakan nyeri di wajah atau sinus? (ya/tidak): tidak
Apakah Anda sensitif terhadap cahaya? (ya/tidak): ya
Apakah Anda mengalami ruam kulit? (ya/tidak): tidak
Apakah Anda merasakan nyeri otot? (ya/tidak): ya
Apakah An